# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HarisBinHabib/Internship-Repo/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
if not os.path.exists("/content/repo"):
    !git clone https://github.com/HarisBinHabib/Internship-Repo.git /content/repo
%cd /content/repo

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
os.makedirs("work/outputs", exist_ok=True)
df.shape

Cloning into '/content/repo'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 134 (delta 46), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (134/134), 1.86 MiB | 6.42 MiB/s, done.
Resolving deltas: 100% (46/46), done.
/content/repo


(30000, 44)

## 1. My rule and its reason codes

**The rule, in plain words:** A page is worth reviewing first if it still gets real traffic
but hasn't been touched in a long time — staleness plus visibility together, not either one
alone.

**Two signals checked before trusting this rule:**

1. **Staleness** (`days_since_last_update >= 180`) — this is the signal behind the starter
   session's `stale_visible_page` refresh flag. Hypothesis: stale pages decline more often
   than fresh pages.
2. **Low CTR at a good position** (`avg_position <= 20` and `ctr < 0.5`) — this is the signal
   behind the starter session's `low_ctr_visible_page` flag. Hypothesis: pages ranking well
   but under-capturing clicks decline more often than pages with healthy CTR at similar
   positions.

Both checks use `trend_direction` only to verify these hypotheses — never as a rule input,
since it's the label-derived column (per the data contract from w03).

**Reason code:** `stale_visible_page`
**Action:** `refresh`

In [2]:
# --- Signal 1: staleness, behind the stale_visible_page flag ---
visible = df[df["impressions_90d"] >= 500].copy()
visible["stale"] = visible["days_since_last_update"] >= 180

bucket1 = visible.groupby("stale").agg(
    n=("trend_direction", "size"),
    decline_rate=("trend_direction", lambda x: (x == "down").mean())
).round(3)
print("Signal 1 — staleness (visible pages only, impressions_90d >= 500):")
print(bucket1)
print(f"\nBase rate (overall decline rate, visible pages): {(visible['trend_direction']=='down').mean():.3f}")

# --- Signal 2: low CTR at a good position, behind the low_ctr_visible_page flag ---
good_position = df[(df["avg_position"] > 0) & (df["avg_position"] <= 20) &
                    (df["impressions_90d"] >= 500)].copy()
good_position["low_ctr"] = good_position["ctr"] < 0.5

bucket2 = good_position.groupby("low_ctr").agg(
    n=("trend_direction", "size"),
    decline_rate=("trend_direction", lambda x: (x == "down").mean())
).round(3)
print("\nSignal 2 — low CTR at good position (avg_position 1-20, impressions_90d >= 500):")
print(bucket2)
print(f"\nBase rate (overall decline rate, good-position pages): {(good_position['trend_direction']=='down').mean():.3f}")



Signal 1 — staleness (visible pages only, impressions_90d >= 500):
           n  decline_rate
stale                     
False  16709         0.595
True      17         0.941

Base rate (overall decline rate, visible pages): 0.596

Signal 2 — low CTR at good position (avg_position 1-20, impressions_90d >= 500):
            n  decline_rate
low_ctr                    
False    2264         0.475
True     9759         0.627

Base rate (overall decline rate, good-position pages): 0.599


**Verdicts:**

- **Signal 1 (staleness):** CONFIRMED — but low-confidence. Stale visible pages decline at
  94.1% (n=17) vs. 59.5% for non-stale visible pages (n=16,709), well above the 59.6% base
  rate. Direction strongly supports the hypothesis, but n=17 is very small — this needs more
  data before I'd trust the magnitude, only the direction.

- **Signal 2 (low CTR at good position):** CONFIRMED. Pages with low CTR at a good position
  decline at 62.7% (n=9,759) vs. 47.5% for pages with healthy CTR at the same position range
  (n=2,264), both compared to a 59.9% base rate. This sample is large enough to trust — low
  CTR at good position is a real, usable signal for this rule.

## 2. Build the ranked queue (writes the CSV)

Score = `impressions_90d` when a page is both stale and visible, 0 otherwise — a readable
score, not a fitted model. Every scored page carries one reason code and one action label.

**Result:** This rule (stale AND visible) is very conservative — only 17 of 30,000 pages
score above zero, because true staleness (`days_since_last_update >= 180`) is rare in this
dataset. Precision@50 collapses to precision@17 in practice (0.941), since there aren't 50
flagged pages to fill the top-50 slot. This is a real limitation, not a bug — discussed
further in Section 4.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
scored = df.copy()

scored["stale"] = (scored["days_since_last_update"] >= 180).astype(int)
scored["visible"] = (scored["impressions_90d"] >= 500).astype(int)

# Transparent score: readable, no fitted weights
scored["score"] = scored["stale"] * scored["visible"] * scored["impressions_90d"]

scored["reason_code"] = "stale_visible_page"
scored["action"] = "refresh"

# Only keep pages the rule actually flags (score > 0) for the review queue
queue = scored[scored["score"] > 0].sort_values("score", ascending=False).reset_index(drop=True)

output_cols = ["content_id", "client_id", "score", "reason_code", "action",
               "impressions_90d", "days_since_last_update", "avg_position", "ctr",
               "trend_direction"]
queue[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Ranked queue written: {len(queue)} pages flagged out of {len(df)} total")
queue[output_cols].head(10)

# Precision@K on the base label, as a sanity check (not the model — just this rule)
k = 50
top_k = queue.head(k)
precision_at_k = (top_k["trend_direction"] == "down").mean()
base_rate = (df["trend_direction"] == "down").mean()
print(f"\nPrecision@{k}: {precision_at_k:.3f}   (base rate for comparison: {base_rate:.3f})")


Ranked queue written: 17 pages flagged out of 30000 total

Precision@50: 0.941   (base rate for comparison: 0.542)


**Top-10 review:**

1. `content_cf56e2e2e282` — Action: refresh. Why: highest score (61,678 impressions, stale
   194 days, declining). What would make it wrong: if a sibling page absorbed this traffic
   (consolidation), refreshing won't help.
2. `content_7368877ea310` — Action: refresh. Why: second-highest impressions (59,472), same
   staleness window, declining. What would make it wrong: same consolidation risk as #1.
3. `content_1bfaa38ff26c` — Action: refresh. Why: 25,715 impressions, CTR 0.23 is reasonable
   for its position (22.2). What would make it wrong: if the decline is seasonal rather than
   content decay — worth checking against site-wide trends first.
4. `content_0a91db491d14` — Action: refresh. Why: decent CTR (0.49) near the position-10
   boundary. What would make it wrong: CTR here is close to acceptable — this may be a weaker
   candidate than its raw impression count suggests.
5. `content_5feee3994adb` — Action: refresh. Why: very low CTR (0.01) despite 7,812
   impressions. What would make it wrong: `avg_position` is 39 — quite deep in the SERP, so a
   refresh may have limited upside regardless of CTR.
6. `content_c2d929d83eaa` — Action: refresh. Why: 7,558 impressions, weak CTR (0.20) at
   position 17.9. What would make it wrong: same consolidation/seasonality checks as above.
7. `content_b16bd7307b39` — Action: refresh. Why: CTR essentially zero (0.00) at position 31.
   What would make it wrong: position 31 is far from page one — refresh alone likely won't
   fix the root visibility problem.
8. `content_fe16a55cd13d` — Action: refresh. Why: CTR 0.33, position 16.4 — a more typical
   mid-tier candidate. What would make it wrong: less clear-cut than the top few; worth a
   human's second look before committing review time.
9. `content_ecb6215e79fd` — Action: refresh. Why: CTR 0.38, position 25.3, moderate volume.
   What would make it wrong: relatively weak signal strength compared to the top of the list.
10. `content_928af3e22c80` — Action: refresh. Why: lowest score in the top 10 (1,697
    impressions), still stale and declining. What would make it wrong: smallest volume here —
    marginal value for the review time it costs.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top10 = queue.head(10)[output_cols]
top10


,content_id,client_id,score,reason_code,action,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
0,content_cf56e2e2e282,client_7f2253d7e2,61678,stale_visible_page,refresh,61678,194,19.7,0.15,down
1,content_7368877ea310,client_7f2253d7e2,59472,stale_visible_page,refresh,59472,194,24.8,0.13,down
2,content_1bfaa38ff26c,client_7f2253d7e2,25715,stale_visible_page,refresh,25715,194,22.2,0.23,down
3,content_0a91db491d14,client_7f2253d7e2,13299,stale_visible_page,refresh,13299,193,10.5,0.49,down
4,content_5feee3994adb,client_7f2253d7e2,7812,stale_visible_page,refresh,7812,194,39.0,0.01,down
5,content_c2d929d83eaa,client_7f2253d7e2,7558,stale_visible_page,refresh,7558,193,17.9,0.20,down
6,content_b16bd7307b39,client_7f2253d7e2,4590,stale_visible_page,refresh,4590,194,31.0,0.00,down
7,content_fe16a55cd13d,client_7f2253d7e2,4556,stale_visible_page,refresh,4556,194,16.4,0.33,down
8,content_ecb6215e79fd,client_7f2253d7e2,4429,stale_visible_page,refresh,4429,194,25.3,0.38,down
9,content_928af3e22c80,client_7f2253d7e2,1697,stale_visible_page,refresh,1697,193,15.8,0.12,down


**Top-10 review:**

1. `content_id` — Action: refresh. Why: high score (stale + visible + N impressions). What
   would make it wrong: if the drop is due to consolidation onto a sibling page, not real decay.
2. ...
(repeat for all 10, using the real content_ids and numbers from the table above)

## 4. Weak picks + leakage check

**Weak picks:** Row 5 (`content_5feee3994adb`) and row 7 (`content_b16bd7307b39`) stand out
as weaker picks — both have `avg_position` in the 30s, well outside typical page-one/page-two
territory. A refresh is a content-quality fix; if the real problem is that these pages rank
too low to be seen at all, refreshing content won't move the needle much. The rule as coded
doesn't factor in position at all, so it can't tell these apart from stronger candidates.

**Bigger limitation:** the rule only flags 17 of 30,000 pages — the `days_since_last_update
>= 180` threshold is rarer in this dataset than expected. A future iteration should test
looser thresholds (e.g. 90 days) to see if the queue grows into something closer to actual
review capacity, without losing precision.

**Leakage check:** confirmed below — the rule's inputs (`days_since_last_update`,
`impressions_90d`) have zero overlap with label-derived columns (`trend_direction`,
`trend_pct`, `is_declining_label`). `trend_direction` was used only to verify Signal 1 and 2
hypotheses in Section 1, never inside the score formula itself.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Explicit leakage check: confirm the score formula never touched trend_direction/trend_pct
rule_inputs = {"days_since_last_update", "impressions_90d"}
label_derived = {"trend_direction", "trend_pct", "is_declining_label"}
print("Rule inputs:", rule_inputs)
print("Overlap with label-derived columns (should be empty):", rule_inputs & label_derived)


Rule inputs: {'impressions_90d', 'days_since_last_update'}
Overlap with label-derived columns (should be empty): set()


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.